In [30]:
import pandas as pd


In [31]:
data = pd.read_csv(
    "/Users/rahulkulkarni/Desktop/personal project/DATA SCIENTIST/online_retail_II.csv",
    encoding="ISO-8859-1",
)
print(data.shape)
print(data.dtypes)
print(data.head())

(1067371, 8)
Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object
  Invoice StockCode                          Description  Quantity  \
0  489434     85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434    79323P                   PINK CHERRY LIGHTS        12   
2  489434    79323W                  WHITE CHERRY LIGHTS        12   
3  489434     22041         RECORD FRAME 7" SINGLE SIZE         48   
4  489434     21232       STRAWBERRY CERAMIC TRINKET BOX        24   

           InvoiceDate  Price  Customer ID         Country  
0  2009-12-01 07:45:00   6.95      13085.0  United Kingdom  
1  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
2  2009-12-01 07:45:00   6.75      13085.0  United Kingdom  
3  2009-12-01 07:45:00   2.10      13085.0  United Kingdom  
4  2009-12-01 07:45:00   1.25      13085.0  United Kingdom  


In [32]:
print(data["Description"].str.contains("Ã|Â|�", na=False, regex=True).sum())


79


# missing values

In [33]:
print(data.isnull().sum())
print(data.isnull().mean() * 100)

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64
Invoice         0.000000
StockCode       0.000000
Description     0.410541
Quantity        0.000000
InvoiceDate     0.000000
Price           0.000000
Customer ID    22.766873
Country         0.000000
dtype: float64


## Date Range Check

In [34]:
data["InvoiceDate"] = pd.to_datetime(data["InvoiceDate"])
print(data["InvoiceDate"].min(), data["InvoiceDate"].max())

2009-12-01 07:45:00 2011-12-09 12:50:00


Q2: Why does the date range matter specifically for calculating "Recency" in RFM — what reference point do you need, and why can't you just use today's real-world date?

Here's the reasoning: Recency is "how many days since this customer's last purchase." That's a relative measurement — it needs a fixed point to count backward from. The natural choice is the max date in the dataset (2011-12-09), or one day after it, used as a stand-in for "today" within the context of this dataset.

If you instead used today's real-world date (2026), every single customer's Recency would be enormous — the "most recent" customer, who bought something on 2011-12-09, would show a Recency of roughly 5,300+ days. That number would be technically correct but completely useless for segmentation, because every customer would look equally "long gone," and you'd lose all ability to distinguish an actively engaged customer from a truly inactive one — the entire point of Recency as a signal would collapse.

So the rule is: Recency's reference point should be relative to the dataset's own timeframe, not the real-world calendar, unless you're specifically building something that updates live against real time (which this static, historical dataset isn't).

## Uniqueness Counts

In [35]:
print("unique customers:", data["Customer ID"].nunique())
print("unique invoices:", data["Invoice"].nunique())
print("unique products (stockcode):", data["StockCode"].nunique())
print("unique countries:", data["Country"].nunique())
print(data["Country"].value_counts().head(10))

unique customers: 5942
unique invoices: 53628
unique products (stockcode): 5305
unique countries: 43
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64


Given that 92% of the data is UK-only and the remaining 8% is spread thin across 42 other countries, would you (a) drop non-UK rows entirely, (b) keep everything and segment globally anyway, or (c) something else?

A3: I'd keep all rows and all real country labels (not collapse non-UK countries into a generic "Other" bucket), and run a single RFM + clustering model across the full customer base — since Country isn't actually used in the RFM calculation itself (Recency/Frequency/Monetary don't depend on it), it doesn't need to affect how the model is built. I would not build separate models per country, since most non-UK countries have too few customers (some in the single digits) to cluster meaningfully on their own. After clustering, I'd report Country as a secondary breakdown — e.g., what % of each segment is UK vs. other countries — and note in the write-up that ~92% of the data is UK-based, so results should be understood as primarily reflecting UK customer behavior.

## Quantity Distribution & Anomalies

In [36]:
print(data["Quantity"].describe())
print("Negative quantity rows:", (data["Quantity"] < 0).sum())
print("Zero quantity rows:", (data["Quantity"] == 0).sum())


count    1.067371e+06
mean     9.938898e+00
std      1.727058e+02
min     -8.099500e+04
25%      1.000000e+00
50%      3.000000e+00
75%      1.000000e+01
max      8.099500e+04
Name: Quantity, dtype: float64
Negative quantity rows: 22950
Zero quantity rows: 0


Q4: The mean (9.94) and median (3.0) are quite different from each other. What does that difference tell you about the shape of this distribution, and why would you generally prefer median over mean as the "typical" value when a dataset looks like this?

A4: The gap between mean and median signals a right-skewed distribution — a small number of extreme outliers (like the 80,995-unit bulk order we found) are pulling the mean upward, even though most transactions are actually small (the median of 3 reflects what a typical order really looks like). This happens because mean is calculated by summing every value and dividing by count, so every value — including extreme outliers — pulls on it equally. Median only depends on position (the middle value when sorted), not magnitude, so it's resistant to outliers. In this dataset, using mean Quantity to describe a "typical customer" would be misleading, since it's distorted by a handful of wholesale-scale orders that don't represent most customers' actual behavior.

## Price Distribution & Anomalies

In [37]:
print(data["Price"].describe())
print("Negative price rows:", (data["Price"] < 0).sum())
print("Zero price rows:", (data["Price"] == 0).sum())


count    1.067371e+06
mean     4.649388e+00
std      1.235531e+02
min     -5.359436e+04
25%      1.250000e+00
50%      2.100000e+00
75%      4.150000e+00
max      3.897000e+04
Name: Price, dtype: float64
Negative price rows: 5
Zero price rows: 6202


Q5: If most of the zero-price rows have a legitimate-looking product description (not "damaged," "test," "lost," etc.) — does that change your decision about whether to drop them? What would you want to check before deciding?

A5: Yes — you can't apply one blanket rule to all zero-price rows, because they're not a single category. Looking at a sample, most zero-price rows fall into two groups: internal warehouse/inventory noise (descriptions like "short," "lost," "damages," "mixed," or unclear codes), and legitimate charge types like DOTCOM POSTAGE priced at $0 in a specific row. Almost all of these also have a missing Customer ID, meaning they'd already be removed by the first cleaning step regardless. The exception is rows with a real product description AND a real Customer ID (e.g., "6 RIBBONS EMPIRE" with Customer ID 16126.0) — these survive the missing-ID filter and need their own explicit decision. Before dropping any zero-price rows, I'd filter specifically to Price == 0 AND Customer ID not null, to see how many rows actually remain after the first cleaning step, rather than assuming the full 6,202 count still applies. Since Monetary is based on real revenue, a $0 transaction contributes nothing to it either way, so dropping the remaining ones is still reasonable — but the count needs to be verified, not assumed.

## cancelled invoices

In [38]:
cancelled = data["Invoice"].astype(str).str.startswith("C")
print("Cancelled invoice rows:", cancelled.sum())
print(cancelled.mean() * 100, "% of rows")

Cancelled invoice rows: 19494
1.8263565339511754 % of rows


You now have three different "negative quantity"/anomaly patterns in this dataset: (1) cancelled invoices — Invoice starts with "C", (2) internal warehouse adjustments — negative quantity + $0 price + no Customer ID, and (3) "Adjust bad debt" rows — StockCode "B", negative price, no Customer ID. If you only used the "C-prefix" rule to filter out negative quantities, what would you miss, and why does that matter for your final cleaned dataset?

A6: Filtering only on the "C" prefix would correctly catch the 19,494 customer-initiated cancellations, but it would miss the ~3,457 internal warehouse-adjustment rows and the 5 "Adjust bad debt" rows, since neither of those patterns is reflected in the invoice number — they'd slide straight through into the "cleaned" dataset undetected. In this specific dataset, it turns out not to matter much in practice, because nearly all of those rows also have a missing Customer ID, so they get removed anyway by the very first cleaning step (dropping missing Customer ID) before the cancellation filter even runs. But that's a coincidence of this particular dataset, not something to rely on generally — if even one adjustment row had a real Customer ID attached, a C-prefix-only filter would silently let bad data corrupt that customer's RFM numbers with no warning. The lesson is not to assume one filter catches everything just because it handles the obvious case — each anomaly pattern needs to be checked independently and verified against what's already being removed, rather than assumed to be covered.

## duplicates

In [40]:
dupes = data[data.duplicated(keep=False)]
print(dupes.sort_values("Invoice").head(20))


    Invoice StockCode                        Description  Quantity  \
362  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
394  489517     21912           VINTAGE SNAKES & LADDERS         1   
391  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
390  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
388  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
386  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
385  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
384  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
379  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
371  489517     21912           VINTAGE SNAKES & LADDERS         1   
368  489517     22130   PARTY CONE CHRISTMAS DECORATION          6   
367  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
365  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
363  489517     2191

You found that "duplicate" rows share the same Invoice number and appear as scattered individual line items rather than the entire invoice duplicated as a block. Does it matter whether you drop these duplicates, and why?

A7: Yes, it matters a lot — and the initial assumption that "exact duplicates should be dropped" turned out to be wrong for this dataset. If duplicates were a true data export glitch, you'd expect the entire invoice to be logged twice, in the same relative order. Instead, individual line items within the same invoice (same Invoice number, same customer, same timestamp) repeat inconsistently — some products appear once, others two or three times. This pattern is more consistent with a customer genuinely adding the same product to their cart multiple times as separate entries, or the system splitting a quantity into multiple rows, rather than an export error. Since these rows represent real purchased units and real revenue, running .drop_duplicates() would silently delete legitimate quantity and undercount that customer's true Frequency and Monetary values. Decision: keep these rows and do not run .drop_duplicates() on the dataset, documenting the investigation and reasoning rather than applying a blanket rule based on an untested assumption.

## Non-Product Stock Codes

In [42]:
non_numeric = data[~data["StockCode"].astype(str).str.match(r"^\d")]
print(non_numeric["StockCode"].value_counts().head(20))


StockCode
POST            2122
DOT             1446
M               1421
C2               282
D                177
S                104
BANK CHARGES     102
ADJUST            67
AMAZONFEE         43
DCGS0058          31
gift_0001_20      29
gift_0001_30      29
DCGSSGIRL         25
DCGSSBOY          23
PADS              19
gift_0001_10      16
CRUK              16
DCGS0076          15
TEST001           15
DCGS0003          14
Name: count, dtype: int64


Given the list of non-product stock codes (POST, DOT, M, C2, D, S, BANK CHARGES, ADJUST, AMAZONFEE, DCGS codes, gift cards, CRUK, TEST001), would you write one blanket rule ("drop everything that's not purely numeric") or would you need multiple, different rules for different code types? What's the risk of using just one simple rule here?

A8: One blanket rule would be a mistake here — the non-numeric codes aren't a single category, they represent at least three different things: legitimate customer-paid revenue (POST, DOT, C2 for postage/carriage; gift_0001_xx for gift cards), pure accounting/administrative noise that should be dropped (M, D, BANK CHARGES, ADJUST, AMAZONFEE, TEST001, CRUK), and ambiguous cases that need direct investigation before deciding (S for samples — likely $0 price and not a real paid transaction; the DCGS-prefixed codes, which don't match the standard numeric pattern but may still be real products with non-standard SKUs). If I dropped everything non-numeric with one rule, I'd incorrectly remove real revenue like postage charges, which would undercount customers' true Monetary values. The risk of a single blanket rule is treating "doesn't match my regex" as equivalent to "not a real transaction," when in reality the pattern only tells you a code is unusual, not why it's unusual — each type needs to be checked individually against its actual meaning before deciding whether it belongs in the cleaned dataset.

## Cleaning

In [43]:
before = len(data)
data = data[data["Customer ID"].notnull()]
print(f"Dropped {before - len(data)} rows -> {len(data)} remaining")


Dropped 243007 rows -> 824364 remaining


In [45]:
before = len(data)
data = data[~data["Invoice"].astype(str).str.startswith("C")]
print(f"Dropped {before - len(data)} rows -> {len(data)} remaining")


Dropped 0 rows -> 805620 remaining


In [46]:
before = len(data)
admin_codes = ["M", "D", "BANK CHARGES", "ADJUST", "AMAZONFEE", "TEST001", "CRUK"]
data = data[~data["StockCode"].isin(admin_codes)]
print(f"Dropped {before - len(data)} rows -> {len(data)} remaining")

Dropped 796 rows -> 804824 remaining


In [47]:
print(
    data[data["StockCode"] == "S"][
        ["StockCode", "Description", "Quantity", "Price"]
    ].head(10)
)


Empty DataFrame
Columns: [StockCode, Description, Quantity, Price]
Index: []


In [48]:
print(
    data[data["StockCode"].astype(str).str.startswith("DCGS")][
        ["StockCode", "Description"]
    ]
    .drop_duplicates()
    .head(10)
)


Empty DataFrame
Columns: [StockCode, Description]
Index: []


In [49]:
print(
    f"FINAL: {len(data)} rows | {data['Customer ID'].nunique()} customers | {data['Invoice'].nunique()} invoices"
)
data.to_csv("cleaned_retail.csv", index=False)


FINAL: 804824 rows | 5855 customers | 36718 invoices
